# Notebook 2 — Headway distributions and leader effects

Run from the `notebooks/` folder. Inputs come from `../data/raw/`; everything this notebook makes is written to `../outputs/`
(`data/`, `tables/`, `figures/`). Figures are PNG at 600 dpi in Cambria with the house colour scheme of `style.py`; tables are Excel in Cambria.
Run the notebooks in order: **1 → 2 → 3**.

Needs `outputs/data/data04.xlsx` from notebook 1. Steps 1 and 2 run 999 refitted bootstrap replicates per family and case (about 15 minutes each).

| Step | Content |
|---|---|
| 1 | Truncated distribution fits (lognormal, log-logistic, Weibull, gamma, generalized gamma) for the four site × target-class cases |
| 2 | Leader-conditioned truncated AFT model; one family for all cases (Weibull); time ratios by leader class |
| 2b | How each leader class shifts the headway distribution relative to the case's main distribution |
| 3 | Off-centred following, occupancy and leader speed as time ratios, crude and leader-adjusted |

## Step 1 — Truncated distribution fitting of time headway (4 cases)

**Cases:** each site (Shahjahanpur, Tikatuli) × target class (PR, BTW), fitted separately. The sites are never merged.

**Window:** events were extracted only when $0.5 \le H \le 5$ s. Above 5 s is free flow, not following, in sub-saturated traffic; below 0.5 s is most likely a coding error. Every likelihood is therefore **truncated** to this window,

$$\ell(\theta)=\sum_i \Big[\log f(H_i;\theta)-\log\{F(5;\theta)-F(0.5;\theta)\}\Big],$$

and every fitted quantity is conditional on following within the window.

**Families:** taken from the headway literature, each written in accelerated-failure-time (AFT) form $H = e^{\mu}Z$ so the same code carries leader conditioning in step 2. Model code is in `hwdist.py`.

| Family | Shape | Notes |
|---|---|---|
| Lognormal | σ | special case of the generalized gamma with Q = 0 |
| Log-logistic | σ | |
| Weibull | σ | special case of the generalized gamma with Q = 1 |
| Gamma | k | special case of the generalized gamma with Q = σ |
| Generalized gamma | σ, Q | Prentice parameterisation |
| Exponential | none | classical random-arrival reference only |

**Checks:** AIC, AICc and BIC with Akaike weights; refitted parametric-bootstrap KS, AD and CvM tests on the PIT values inside the window; likelihood-ratio tests of the three nested families against the generalized gamma; observed vs fitted short- and long-headway shares.

**Outputs:** `outputs/tables/01_truncated_distribution_fit.xlsx`, plus figures `truncated_fit_density.png` and `truncated_fit_cdf_difference.png`.

In [ ]:
import sys; sys.path.insert(0, str(__import__('pathlib').Path.cwd()))
from style import PAL, use_style
from pathlib import Path
import hashlib, json, time
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from openpyxl import load_workbook
from openpyxl.styles import Font
import hwdist as hd

HERE = Path.cwd()
DATA = HERE.parent / "outputs" / "data" / "data04.xlsx"
TAB = HERE.parent / "outputs" / "tables"; TAB.mkdir(parents=True, exist_ok=True)
FIG = HERE.parent / "outputs" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
FONT = "Cambria"
plt.rcParams.update({"font.family": FONT, "mathtext.fontset": "custom", "mathtext.rm": FONT,
                     "mathtext.it": FONT + ":italic", "mathtext.bf": FONT + ":bold", "font.size": 9})
def save_fig(fig, name):
    p = FIG / f"{name}.png"; p.unlink(missing_ok=True); fig.savefig(p, dpi=600, bbox_inches="tight"); print("saved", p.name)
def cambria_xlsx(path):
    wb = load_workbook(path)
    for ws in wb.worksheets:
        for row in ws.iter_rows():
            for c in row:
                c.font = Font(name=FONT, bold=(c.row == 1), size=11)
    wb.save(path)

### Settings

In [ ]:
LOWER, UPPER = 0.5, 5.0            # s, extraction window (truncation bounds)
N_BOOT = 999                       # refitted parametric-bootstrap replicates per case x family
SEED = 20261106
CASES = [("Shahjahanpur", "PR"), ("Shahjahanpur", "BTW"), ("Tikatuli", "PR"), ("Tikatuli", "BTW")]
FAMS = hd.FAMILIES                 # 5 candidates + exponential reference
CANDIDATES = [f for f in FAMS if f != "exponential"]
FAM_COL = PAL["family"]

### Data

In [ ]:
d = pd.read_excel(DATA, sheet_name="data04")
print("data04:", d.shape, " sha256:", hashlib.sha256(DATA.read_bytes()).hexdigest()[:16])
assert d.Time_Headway_s.between(LOWER, UPPER).all(), "observation outside the extraction window"
data = {c: d[(d.Site == c[0]) & (d.Target_Class == c[1])].Time_Headway_s.to_numpy(float) for c in CASES}
print({f"{s} {t}": len(x) for (s, t), x in data.items()})

### Fit all families and bootstrap the goodness of fit

In [ ]:
fits, rows = {}, []
t0 = time.time()
for ci, case in enumerate(CASES):
    x = data[case]
    for fi, fam in enumerate(FAMS):
        r = hd.fit(x, fam, lower=LOWER, upper=UPPER); fits[(case, fam)] = r
        obs, p, valid = hd.bootstrap_gof(x, r, B=N_BOOT, seed=SEED + 100 * ci + fi)
        mu = r["beta"][0]; sh = r["shape"]
        q = hd.ppf(hd.cdf(LOWER, mu, fam, sh) + np.array([0.25, 0.5, 0.75]) *
                   (hd.cdf(UPPER, mu, fam, sh) - hd.cdf(LOWER, mu, fam, sh)), mu, fam, sh)
        grid = np.linspace(LOWER, UPPER, 4001)
        mean_w = np.trapezoid(grid * hd.trunc_pdf(grid, mu, fam, sh, LOWER, UPPER), grid)
        row = {"Site": case[0], "Class": case[1], "Family": hd.LABELS[fam], "family_key": fam, "n": r["n"], "n_params": r["k"],
               "mu": mu, **{("k_shape" if nm == "k" else nm): v for nm, v in zip(hd.SHAPE_NAMES[fam], sh)},
               "logLik": r["loglik"], "AIC": r["AIC"], "AICc": r["AICc"], "BIC": r["BIC"],
               "fitted_mean_in_window_s": mean_w, "fitted_Q1_s": q[0], "fitted_median_s": q[1], "fitted_Q3_s": q[2],
               "obs_mean_s": x.mean(), "obs_median_s": np.median(x),
               "KS_D": obs[0], "KS_p": p[0], "AD_A2": obs[1], "AD_p": p[1], "CvM_W2": obs[2], "CvM_p": p[2],
               "boot_valid": valid}
        for t in (1.5, 2.0):
            row[f"obs_P(H<{t})"] = (x < t).mean(); row[f"fit_P(H<{t})"] = hd.trunc_cdf(t, mu, fam, sh, LOWER, UPPER)
        row["obs_P(H>4.0)"] = (x > 4.0).mean(); row["fit_P(H>4.0)"] = 1 - hd.trunc_cdf(4.0, mu, fam, sh, LOWER, UPPER)
        rows.append(row)
    print(f"{case} done  ({time.time() - t0:.0f} s)")
res = pd.DataFrame(rows)
cand = res.family_key.isin(CANDIDATES)
res["dAIC"] = np.nan; res["Akaike_weight"] = np.nan
for case, g in res[cand].groupby(["Site", "Class"]):
    da = g.AIC - g.AIC.min(); w = np.exp(-da / 2); w /= w.sum()
    res.loc[g.index, "dAIC"] = da; res.loc[g.index, "Akaike_weight"] = w
res.loc[~cand, "dAIC"] = res[~cand].AIC.values - res[cand].groupby(["Site", "Class"]).AIC.min().reindex(
    list(zip(res[~cand].Site, res[~cand].Class))).values
res["adequate_KS_AD_CvM"] = (res[["KS_p", "AD_p", "CvM_p"]] >= 0.05).all(axis=1)

### Ranking and adequacy by case

In [ ]:
cols = ["Site", "Class", "Family", "n_params", "logLik", "AIC", "dAIC", "Akaike_weight", "KS_p", "AD_p", "CvM_p", "adequate_KS_AD_CvM"]
for case, g in res.groupby(["Site", "Class"], sort=False):
    print(g.sort_values("AIC")[cols].round(3).to_string(index=False)); print()

### Parameters and fitted vs observed summaries (inside the window)

In [ ]:
pcols = ["Site", "Class", "Family", "mu", "sigma", "k_shape", "Q", "fitted_median_s", "obs_median_s", "fitted_mean_in_window_s", "obs_mean_s",
         "obs_P(H<1.5)", "fit_P(H<1.5)", "obs_P(H<2.0)", "fit_P(H<2.0)", "obs_P(H>4.0)", "fit_P(H>4.0)"]
print(res[res.family_key.isin(CANDIDATES)].reindex(columns=pcols).round(3).to_string(index=False))

### Nested likelihood-ratio tests against the generalized gamma

$Q=0$ gives the lognormal, $Q=1$ the Weibull and $Q=\sigma$ the gamma. Each test has 1 df (interior restriction). A small p-value means the two-parameter family is rejected in favour of the three-parameter generalized gamma.

In [ ]:
lr = []
for case in CASES:
    llg = fits[(case, "gengamma")]["loglik"]; Qhat = fits[(case, "gengamma")]["shape"][1]
    for fam in ["lognormal", "weibull", "gamma"]:
        stat = max(2 * (llg - fits[(case, fam)]["loglik"]), 0.0)
        lr.append({"Site": case[0], "Class": case[1], "Restricted": hd.LABELS[fam], "GG_Q_hat": Qhat,
                   "LR_stat": stat, "df": 1, "p": stats.chi2.sf(stat, 1)})
lr = pd.DataFrame(lr)
print(lr.round(4).to_string(index=False))

### Figure: fitted truncated densities

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(7.2, 6.0), sharex=True)
grid = np.linspace(LOWER, UPPER, 400); bins = np.arange(LOWER, UPPER + 0.25, 0.25)
for ax, case, tag in zip(axes.ravel(), CASES, "abcd"):
    x = data[case]
    ax.hist(x, bins=bins, density=True, color="0.82", edgecolor="w", lw=.4)
    g = res[(res.Site == case[0]) & (res.Class == case[1])]
    for fam in FAMS:
        r = fits[(case, fam)]; dA = g[g.family_key == fam].dAIC.iloc[0]
        ax.plot(grid, hd.trunc_pdf(grid, r["beta"][0], fam, r["shape"], LOWER, UPPER), color=FAM_COL[fam],
                lw=1.0 if fam == "exponential" else 1.4, ls="--" if fam == "exponential" else "-",
                label=f"{hd.LABELS[fam]} ($\\Delta$AIC {dA:.1f})")
    ax.set_title(f"({tag}) {case[0]}, {case[1]} (n={len(x)})", loc="left", fontsize=8.5)
    ax.legend(frameon=False, fontsize=6.3, loc="upper right"); ax.grid(alpha=.25, lw=.5)
for ax in axes[1]: ax.set_xlabel("Time headway (s)")
for ax in axes[:, 0]: ax.set_ylabel("Density (truncated to 0.5–5 s)")
fig.tight_layout(); save_fig(fig, "truncated_fit_density"); plt.show()

### Figure: where each family misfits

Empirical CDF minus fitted truncated CDF. A family that fits stays near zero everywhere. The grey band is the approximate 95% KS band for a fully specified model, $\pm 1.36/\sqrt{n}$, shown only for orientation; the bootstrap p-values in the table account for estimation.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(7.2, 6.0), sharex=True)
for ax, case, tag in zip(axes.ravel(), CASES, "abcd"):
    x = np.sort(data[case]); n = len(x); ecdf = np.arange(1, n + 1) / n
    ax.axhspan(-1.36 / np.sqrt(n), 1.36 / np.sqrt(n), color="0.9", zorder=0)
    ax.axhline(0, color="0.4", lw=.7)
    for fam in CANDIDATES:
        r = fits[(case, fam)]
        ax.plot(x, ecdf - hd.trunc_cdf(x, r["beta"][0], fam, r["shape"], LOWER, UPPER), color=FAM_COL[fam], lw=1.2,
                label=hd.LABELS[fam])
    ax.set_title(f"({tag}) {case[0]}, {case[1]} (n={n})", loc="left", fontsize=8.5)
    ax.grid(alpha=.25, lw=.5)
axes[0, 0].legend(frameon=False, fontsize=6.8, loc="lower left", ncol=2)
for ax in axes[1]: ax.set_xlabel("Time headway (s)")
for ax in axes[:, 0]: ax.set_ylabel("Empirical − fitted CDF")
fig.tight_layout(); save_fig(fig, "truncated_fit_cdf_difference"); plt.show()

### Export

In [ ]:
out = TAB / "01_truncated_distribution_fit.xlsx"
summary = (res[res.family_key.isin(CANDIDATES)].sort_values(["Site", "Class", "AIC"])
           .groupby(["Site", "Class"], sort=False)
           .apply(lambda g: pd.Series({"Best_by_AIC": g.Family.iloc[0], "Within_2_AIC": ", ".join(g[g.dAIC < 2].Family),
                                       "Adequate_families": ", ".join(g[g.adequate_KS_AD_CvM].Family) or "none"}),
                  include_groups=False).reset_index())
print(summary.to_string(index=False))
with pd.ExcelWriter(out) as xw:
    summary.to_excel(xw, sheet_name="summary", index=False)
    res.drop(columns="family_key").to_excel(xw, sheet_name="fits_gof", index=False)
    lr.to_excel(xw, sheet_name="nested_LR", index=False)
    pd.DataFrame({"Parameter": ["LOWER", "UPPER", "N_BOOT", "SEED", "families", "data04_sha256"],
                  "Value": [LOWER, UPPER, N_BOOT, SEED, json.dumps(FAMS), hashlib.sha256(DATA.read_bytes()).hexdigest()]}
                 ).to_excel(xw, sheet_name="settings", index=False)
cambria_xlsx(out); print("wrote", out.name)

## Step 2 — Leader-conditioned AFT models of time headway (4 cases)

**Model:** for each site × target class case and each family from step 1, the location of the truncated AFT model depends only on the class of the leader,

$$\log H_i = \beta_0 + \sum_{c \ne \text{4W}} \beta_c\,\mathbb{1}[\text{leader}_i = c] + \sigma\,\varepsilon_i,\qquad 0.5 \le H_i \le 5\ \text{s},$$

with five leader classes (4W, BTW, CNG, MT_2W, NMT) and 4W as the reference. $e^{\beta_c}$ is a **time ratio**: the factor by which the whole headway distribution is stretched (> 1) or compressed (< 1) when following leader class $c$ instead of a four-wheeler. The shape is shared across leaders, so the leader effect is a pure scale shift. With n = 77 in the smallest case, nothing beyond leader class is fitted here; other variables are added one at a time in step 3.

**Truncation:** the likelihood is truncated to the 0.5–5 s extraction window, exactly as in step 1, so every result is conditional on following within the window.

**Steps:**
1. Fit the six families (five candidates + exponential reference) with and without leader class.
2. Likelihood-ratio test of leader class (4 df) and ΔAIC against the unconditional fit of step 1.
3. Choose one family for all four cases from the conditional fits (summed AIC across cases, with per-case ΔAIC and adequacy shown).
4. Check adequacy: refitted parametric-bootstrap KS, AD and CvM on the conditional PIT values, and Cox–Snell residual plots.
5. Report time ratios with 95% Wald CIs and model-based median headway by leader class.

**Outputs:** `outputs/tables/02_leader_conditioned_aft.xlsx`, and figures `leader_aft_time_ratios.png`, `leader_aft_cox_snell.png`.

In [ ]:
import sys; sys.path.insert(0, str(__import__('pathlib').Path.cwd()))
from style import PAL, use_style
from pathlib import Path
import hashlib, json, time
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from openpyxl import load_workbook
from openpyxl.styles import Font
import hwdist as hd

HERE = Path.cwd()
DATA = HERE.parent / "outputs" / "data" / "data04.xlsx"
TAB = HERE.parent / "outputs" / "tables"; TAB.mkdir(parents=True, exist_ok=True)
FIG = HERE.parent / "outputs" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
FONT = "Cambria"
plt.rcParams.update({"font.family": FONT, "mathtext.fontset": "custom", "mathtext.rm": FONT,
                     "mathtext.it": FONT + ":italic", "mathtext.bf": FONT + ":bold", "font.size": 9})
def save_fig(fig, name):
    p = FIG / f"{name}.png"; p.unlink(missing_ok=True); fig.savefig(p, dpi=600, bbox_inches="tight"); print("saved", p.name)
def cambria_xlsx(path):
    wb = load_workbook(path)
    for ws in wb.worksheets:
        for row in ws.iter_rows():
            for c in row:
                c.font = Font(name=FONT, bold=(c.row == 1), size=11)
    wb.save(path)

### Settings

In [ ]:
LOWER, UPPER = 0.5, 5.0            # s, extraction window (truncation bounds)
N_BOOT = 999                       # refitted parametric-bootstrap replicates per case x family
SEED = 20261107
CASES = [("Shahjahanpur", "PR"), ("Shahjahanpur", "BTW"), ("Tikatuli", "PR"), ("Tikatuli", "BTW")]
LEADERS = ["4W", "BTW", "CNG", "MT_2W", "NMT"]
REF = "4W"                         # reference leader class (largest class in 3 of 4 cases)
FAMS = hd.FAMILIES
CANDIDATES = [f for f in FAMS if f != "exponential"]
LEADER_COL = PAL["leader"]

### Data and design matrices

In [ ]:
d = pd.read_excel(DATA, sheet_name="data04")
print("data04:", d.shape, " sha256:", hashlib.sha256(DATA.read_bytes()).hexdigest()[:16])
assert d.Time_Headway_s.between(LOWER, UPPER).all()
assert set(d.Leader_Class) == set(LEADERS)
NONREF = [c for c in LEADERS if c != REF]
data = {}
for case in CASES:
    g = d[(d.Site == case[0]) & (d.Target_Class == case[1])]
    X = np.column_stack([np.ones(len(g))] + [(g.Leader_Class == c).to_numpy(float) for c in NONREF])
    data[case] = (g.Time_Headway_s.to_numpy(float), X, g.Leader_Class.to_numpy())
print(pd.crosstab([d.Site, d.Target_Class], d.Leader_Class).reindex(columns=LEADERS))

### Fit every family with and without leader class

In [ ]:
fits0, fits1, rows = {}, {}, []
for case in CASES:
    x, X, _ = data[case]
    for fam in FAMS:
        r0 = hd.fit(x, fam, lower=LOWER, upper=UPPER)
        r1 = hd.fit(x, fam, X=X, lower=LOWER, upper=UPPER)
        if r1["loglik"] < r0["loglik"]:        # safeguard: start the conditional fit from the nested solution
            r1 = hd.fit(x, fam, X=X, lower=LOWER, upper=UPPER, start=np.r_[r0["beta"], np.zeros(len(NONREF)), r0["theta"][1:]])
        fits0[(case, fam)], fits1[(case, fam)] = r0, r1
        lr = max(2 * (r1["loglik"] - r0["loglik"]), 0.0); df = len(NONREF)
        rows.append({"Site": case[0], "Class": case[1], "Family": hd.LABELS[fam], "family_key": fam, "n": r1["n"],
                     "n_params_uncond": r0["k"], "n_params_leader": r1["k"],
                     "logLik_uncond": r0["loglik"], "logLik_leader": r1["loglik"],
                     "AIC_uncond": r0["AIC"], "AIC_leader": r1["AIC"], "AICc_leader": r1["AICc"], "BIC_leader": r1["BIC"],
                     "AIC_change": r1["AIC"] - r0["AIC"], "LR_stat": lr, "LR_df": df, "LR_p": stats.chi2.sf(lr, df),
                     **{("k_shape" if nm == "k" else nm): v for nm, v in zip(hd.SHAPE_NAMES[fam], r1["shape"])}})
comp = pd.DataFrame(rows)
cand = comp.family_key.isin(CANDIDATES)
comp["dAIC_leader"] = np.nan; comp["Akaike_weight_leader"] = np.nan
for _, g in comp[cand].groupby(["Site", "Class"]):
    da = g.AIC_leader - g.AIC_leader.min(); w = np.exp(-da / 2); w /= w.sum()
    comp.loc[g.index, "dAIC_leader"] = da; comp.loc[g.index, "Akaike_weight_leader"] = w
cols = ["Site", "Class", "Family", "AIC_uncond", "AIC_leader", "AIC_change", "LR_stat", "LR_p", "dAIC_leader", "Akaike_weight_leader"]
for _, g in comp.groupby(["Site", "Class"], sort=False):
    print(g.sort_values("AIC_leader")[cols].round(4).to_string(index=False)); print()

### Choice of one family for all four cases

The same family is used in every case so the time ratios are comparable across sites and target classes.

**Rule (parsimony with a nested check):**
1. Rank the families by leader-conditioned AIC summed over the four cases (equivalent to a joint model with case-specific parameters).
2. Lognormal (Q = 0), Weibull (Q = 1) and gamma (Q = σ) are nested in the generalized gamma. For each, a joint likelihood-ratio test against the generalized gamma sums the per-case LR statistics (4 df).
3. The chosen family is the best-ranked two-parameter family that the joint test does not reject at 5%. The generalized gamma is used only if every nested family is rejected.

The extra shape parameter is not carried unless the data demand it. In a small case it can also sit on a weakly identified ridge (|Q| large, σ near 0), so the fitted Q is shown.

In [ ]:
gg_ll = {case: fits1[(case, "gengamma")]["loglik"] for case in CASES}
gg_Q = {case: fits1[(case, "gengamma")]["shape"][1] for case in CASES}
comp["GG_Q_leader"] = [gg_Q[(a, b)] for a, b in zip(comp.Site, comp.Class)]
comp["LR_vs_GG"] = [max(2 * (gg_ll[(a, b)] - ll), 0.0) if k in ("lognormal", "weibull", "gamma") else np.nan
                    for a, b, k, ll in zip(comp.Site, comp.Class, comp.family_key, comp.logLik_leader)]
comp["LR_vs_GG_p"] = stats.chi2.sf(comp.LR_vs_GG, 1)
print("generalized gamma Q (leader-conditioned):", {f"{a} {b}": round(q, 3) for (a, b), q in gg_Q.items()})
choice = (comp[cand].groupby("Family")
          .agg(family_key=("family_key", "first"), sum_AIC_leader=("AIC_leader", "sum"), max_dAIC_in_a_case=("dAIC_leader", "max"),
               cases_best=("dAIC_leader", lambda s: int((s == 0).sum())), cases_within_2=("dAIC_leader", lambda s: int((s < 2).sum())),
               joint_LR_vs_GG=("LR_vs_GG", lambda s: s.sum() if s.notna().all() else np.nan))
          .sort_values("sum_AIC_leader"))
choice["n_shape"] = [hd.N_SHAPE[k] for k in choice.family_key]
choice["d_sum_AIC"] = choice.sum_AIC_leader - choice.sum_AIC_leader.min()
choice["joint_LR_df"] = np.where(choice.joint_LR_vs_GG.notna(), len(CASES), np.nan)
choice["joint_LR_p"] = stats.chi2.sf(choice.joint_LR_vs_GG, len(CASES))
choice["rejected_by_GG"] = choice.joint_LR_p < 0.05
choice = choice.reset_index()
print(choice.drop(columns="family_key").round(4).to_string(index=False))
two = choice[(choice.n_shape == 1) & ~choice.rejected_by_GG].sort_values("sum_AIC_leader")
CHOSEN = two.family_key.iloc[0] if len(two) else "gengamma"
print("\nchosen family:", hd.LABELS[CHOSEN])

### Adequacy of the conditional fits: refitted parametric bootstrap on PIT values

In [ ]:
gof = []; t0 = time.time()
for ci, case in enumerate(CASES):
    x, X, _ = data[case]
    for fi, fam in enumerate(CANDIDATES):
        obs, p, valid = hd.bootstrap_gof(x, fits1[(case, fam)], B=N_BOOT, seed=SEED + 100 * ci + fi)
        gof.append({"Site": case[0], "Class": case[1], "Family": hd.LABELS[fam], "KS_D": obs[0], "KS_p": p[0],
                    "AD_A2": obs[1], "AD_p": p[1], "CvM_W2": obs[2], "CvM_p": p[2], "boot_valid": valid})
    print(f"{case} done ({time.time() - t0:.0f} s)")
gof = pd.DataFrame(gof)
gof["adequate_KS_AD_CvM"] = (gof[["KS_p", "AD_p", "CvM_p"]] >= 0.05).all(axis=1)
comp = comp.merge(gof, on=["Site", "Class", "Family"], how="left")
print(gof.round(3).to_string(index=False))

### Time ratios by leader class (all candidate families)

$e^{\hat\beta_c}$ with 95% Wald intervals from the inverse observed information. The chosen family is reported in the manuscript; the others show how sensitive the leader effects are to the family.

In [ ]:
tr = []
for case in CASES:
    x, X, lead = data[case]
    for fam in CANDIDATES:
        r = fits1[(case, fam)]; V = hd.covariance(r, x); se = np.sqrt(np.clip(np.diag(V)[:X.shape[1]], 0, None))
        for j, c in enumerate(NONREF, start=1):
            b = r["beta"][j]
            tr.append({"Site": case[0], "Class": case[1], "Family": hd.LABELS[fam], "Leader": c, "n_leader": int((lead == c).sum()),
                       "n_ref": int((lead == REF).sum()), "beta": b, "SE": se[j], "z": b / se[j], "p_Wald": 2 * stats.norm.sf(abs(b / se[j])),
                       "TR": np.exp(b), "TR_lo95": np.exp(b - 1.96 * se[j]), "TR_hi95": np.exp(b + 1.96 * se[j])})
tr = pd.DataFrame(tr)
print(tr[tr.Family == hd.LABELS[CHOSEN]].round(3).to_string(index=False))

### Model-based headway by leader class (chosen family)

Median and mean of the fitted truncated distribution for each leader class, beside the observed values. Because the window truncates the distribution, the fitted median ratio between two leaders is close to, but not exactly, the time ratio.

In [ ]:
grid = np.linspace(LOWER, UPPER, 4001); med = []
for case in CASES:
    x, X, lead = data[case]; r = fits1[(case, CHOSEN)]
    for j, c in enumerate(LEADERS):
        mu = r["beta"][0] + (r["beta"][NONREF.index(c) + 1] if c != REF else 0.0)
        Fa, Fb = hd.cdf(LOWER, mu, CHOSEN, r["shape"]), hd.cdf(UPPER, mu, CHOSEN, r["shape"])
        fmed = hd.ppf(Fa + 0.5 * (Fb - Fa), mu, CHOSEN, r["shape"])
        fmean = np.trapezoid(grid * hd.trunc_pdf(grid, mu, CHOSEN, r["shape"], LOWER, UPPER), grid)
        xs = x[lead == c]
        med.append({"Site": case[0], "Class": case[1], "Leader": c, "n": len(xs), "obs_median_s": np.median(xs), "fit_median_s": fmed,
                    "obs_mean_s": xs.mean(), "fit_mean_s": fmean, "obs_P(H<2.0)": (xs < 2).mean(),
                    "fit_P(H<2.0)": hd.trunc_cdf(2.0, mu, CHOSEN, r["shape"], LOWER, UPPER)})
med = pd.DataFrame(med)
print(med.round(3).to_string(index=False))

### Figure: time ratios by leader class (chosen family)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(7.4, 2.9), sharey=True, sharex=True)
sub = tr[tr.Family == hd.LABELS[CHOSEN]]
ypos = np.arange(len(NONREF))[::-1]
for ax, case, tag in zip(axes, CASES, "abcd"):
    g = sub[(sub.Site == case[0]) & (sub.Class == case[1])].set_index("Leader").loc[NONREF]
    ax.axvline(1, color="0.4", lw=.7)
    for y, c in zip(ypos, NONREF):
        ax.plot([g.loc[c, "TR_lo95"], g.loc[c, "TR_hi95"]], [y, y], color=LEADER_COL[c], lw=1.4)
        ax.plot(g.loc[c, "TR"], y, "o", color=LEADER_COL[c], ms=4)
    ax.set_title(f"({tag}) {case[0]}, {case[1]}\n(n={len(data[case][0])})", loc="left", fontsize=8)
    ax.set_xscale("log"); ax.set_xticks([0.5, 0.7, 1, 1.4]); ax.set_xticklabels(["0.5", "0.7", "1", "1.4"])
    ax.minorticks_off(); ax.grid(axis="x", alpha=.25, lw=.5)
axes[0].set_yticks(ypos); axes[0].set_yticklabels(NONREF)
axes[0].set_ylabel("Leader class (reference: 4W)")
fig.supxlabel(f"Time ratio vs 4W leader, {hd.LABELS[CHOSEN]} AFT (95% CI, log scale)", fontsize=9)
fig.tight_layout(); save_fig(fig, "leader_aft_time_ratios"); plt.show()

### Figure: Cox–Snell residuals (chosen family)

$r_i = -\log\{1-\hat F_{\text{trunc}}(H_i \mid \text{leader}_i)\}$. If the conditional model is right, the $r_i$ are a unit-exponential sample, so their Nelson–Aalen cumulative hazard follows the 45° line.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(6.0, 5.6))
for ax, case, tag in zip(axes.ravel(), CASES, "abcd"):
    x, X, _ = data[case]; u = np.clip(hd.pit(x, fits1[(case, CHOSEN)]), 1e-12, 1 - 1e-12)
    rcs = np.sort(-np.log1p(-u)); n = len(rcs); na = np.cumsum(1 / (n - np.arange(n)))
    m = max(rcs.max(), na.max()) * 1.03
    ax.plot([0, m], [0, m], color="0.4", lw=.8, ls="--")
    ax.step(rcs, na, where="post", color=PAL["family"][CHOSEN], lw=1.2)
    ax.set_xlim(0, m); ax.set_ylim(0, m); ax.set_aspect("equal")
    ax.set_title(f"({tag}) {case[0]}, {case[1]} (n={n})", loc="left", fontsize=8.5); ax.grid(alpha=.25, lw=.5)
for ax in axes[1]: ax.set_xlabel("Cox–Snell residual")
for ax in axes[:, 0]: ax.set_ylabel("Nelson–Aalen cumulative hazard")
fig.tight_layout(); save_fig(fig, "leader_aft_cox_snell"); plt.show()

### Export

In [ ]:
out = TAB / "02_leader_conditioned_aft.xlsx"
ch = comp[comp.family_key == CHOSEN]
summary = ch[["Site", "Class", "n", "AIC_uncond", "AIC_leader", "AIC_change", "LR_stat", "LR_df", "LR_p", "dAIC_leader",
              "KS_p", "AD_p", "CvM_p", "adequate_KS_AD_CvM"]].copy()
summary.insert(2, "Family", hd.LABELS[CHOSEN])
summary["Best_conditional_by_AIC"] = [comp[(comp.Site == s) & (comp.Class == c) & cand].sort_values("AIC_leader").Family.iloc[0]
                                      for s, c in zip(summary.Site, summary.Class)]
print(summary.round(4).to_string(index=False))
with pd.ExcelWriter(out) as xw:
    summary.to_excel(xw, sheet_name="summary", index=False)
    choice.drop(columns="family_key").to_excel(xw, sheet_name="family_choice", index=False)
    comp.drop(columns="family_key").to_excel(xw, sheet_name="fits_LR_gof", index=False)
    tr[tr.Family == hd.LABELS[CHOSEN]].to_excel(xw, sheet_name="time_ratios", index=False)
    tr.to_excel(xw, sheet_name="time_ratios_all_families", index=False)
    med.to_excel(xw, sheet_name="headway_by_leader", index=False)
    pd.DataFrame({"Parameter": ["LOWER", "UPPER", "N_BOOT", "SEED", "families", "leader_classes", "reference_leader", "chosen_family",
                                "data04_sha256"],
                  "Value": [LOWER, UPPER, N_BOOT, SEED, json.dumps(FAMS), json.dumps(LEADERS), REF, hd.LABELS[CHOSEN],
                            hashlib.sha256(DATA.read_bytes()).hexdigest()]}).to_excel(xw, sheet_name="settings", index=False)
cambria_xlsx(out); print("wrote", out.name)

## Step 2b — How each leader class shifts the headway distribution (4 cases)

Notebook 02 gives time ratios against a four-wheeler leader. Here every leader class is compared with the **main distribution** of its case, which is the headway distribution of all events in that site × target class. The comparison answers two questions.

**1. Type of effect.** In the leader-conditioned Weibull AFT, a leader can only **shift** the distribution along the time axis, compressing it (shorter headways) or stretching it (longer headways). The shape is held the same. To check that assumption, the model is extended so the Weibull shape σ also depends on the leader, which would let a leader make headways more or less **dispersed** as well. A likelihood-ratio test (4 df) per case says whether that extra freedom is needed.

**2. Extent of effect.** For each leader class, against the main distribution:
- the median ratio (leader median ÷ main median, close to a time ratio against the case average);
- the change in median and mean headway (s and %);
- the change in the share of short headways, P(H < 2 s), in percentage points;
- the Kolmogorov distance between the two distributions (maximum CDF gap).

Under the same fitted model, the main distribution is the mixture of the five leader-specific distributions, weighted by the observed leader shares. This keeps leader and main on one consistent footing. It is checked against the unconditional Weibull fit of step 1.

**Uncertainty:** 95% intervals from 4,000 draws of the parameter vector from its asymptotic normal distribution (inverse observed information).

**Classification:** compresses if the median-ratio interval lies wholly below 1; stretches if it lies wholly above 1; otherwise no clear shift.

**Outputs:** `outputs/tables/02b_leader_distribution_shifts.xlsx`, and figures `leader_shifted_densities.png`, `leader_shift_vs_main.png`.

In [ ]:
import sys; sys.path.insert(0, str(__import__('pathlib').Path.cwd()))
from style import PAL, use_style
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
from scipy import stats, optimize
import matplotlib.pyplot as plt
from openpyxl import load_workbook
from openpyxl.styles import Font
import hwdist as hd

HERE = Path.cwd()
DATA = HERE.parent / "outputs" / "data" / "data04.xlsx"
TAB = HERE.parent / "outputs" / "tables"; TAB.mkdir(parents=True, exist_ok=True)
FIG = HERE.parent / "outputs" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
FONT = "Cambria"
plt.rcParams.update({"font.family": FONT, "mathtext.fontset": "custom", "mathtext.rm": FONT,
                     "mathtext.it": FONT + ":italic", "mathtext.bf": FONT + ":bold", "font.size": 9})
def save_fig(fig, name):
    p = FIG / f"{name}.png"; p.unlink(missing_ok=True); fig.savefig(p, dpi=600, bbox_inches="tight"); print("saved", p.name)
def cambria_xlsx(path):
    wb = load_workbook(path)
    for ws in wb.worksheets:
        for row in ws.iter_rows():
            for c in row:
                c.font = Font(name=FONT, bold=(c.row == 1), size=11)
    wb.save(path)

### Settings

In [ ]:
LOWER, UPPER = 0.5, 5.0
FAMILY = "weibull"                 # chosen in notebook 02 (checked below)
_chosen02 = pd.read_excel(TAB / "02_leader_conditioned_aft.xlsx", sheet_name="settings").set_index("Parameter").Value["chosen_family"]
assert _chosen02 == hd.LABELS[FAMILY], f"notebook 02 chose {_chosen02}"
CASES = [("Shahjahanpur", "PR"), ("Shahjahanpur", "BTW"), ("Tikatuli", "PR"), ("Tikatuli", "BTW")]
LEADERS = ["4W", "BTW", "CNG", "MT_2W", "NMT"]; REF = "4W"
NONREF = [c for c in LEADERS if c != REF]
SHORT = 2.0                        # s, short-headway threshold
N_DRAW = 4000; SEED = 20261107
LEADER_COL = PAL["leader"]
GRID = np.linspace(LOWER, UPPER, 2001)

### Data

In [ ]:
d = pd.read_excel(DATA, sheet_name="data04")
print("data04:", d.shape, " sha256:", hashlib.sha256(DATA.read_bytes()).hexdigest()[:16])
data = {}
for case in CASES:
    g = d[(d.Site == case[0]) & (d.Target_Class == case[1])]
    X = np.column_stack([np.ones(len(g))] + [(g.Leader_Class == c).to_numpy(float) for c in NONREF])
    share = g.Leader_Class.value_counts(normalize=True).reindex(LEADERS).to_numpy()
    data[case] = (g.Time_Headway_s.to_numpy(float), X, g.Leader_Class.to_numpy(), share)

### Type of effect: does the leader also change the spread?

Base model: location depends on leader, one shared σ (step 2). Extended model: $\log\sigma_i = \alpha_0 + \sum_c \alpha_c\,\mathbb{1}[\text{leader}_i=c]$ as well. If the LR test is not significant, the leader effect is a pure shift (compression or stretch) of a common shape.

In [ ]:
def nll_het(theta, x, X):
    p = X.shape[1]; mu = X @ theta[:p]; s = np.exp(X @ theta[p:])
    with np.errstate(all="ignore"):
        lf = hd.logpdf(x, mu, FAMILY, [s])
        Z = hd.cdf(UPPER, mu, FAMILY, [s]) - hd.cdf(LOWER, mu, FAMILY, [s])
        if not np.all(np.isfinite(Z)) or Z.min() < 1e-10:
            return 1e12
        v = -(lf.sum() - np.log(Z).sum())
    return v if np.isfinite(v) else 1e12

fits, spread = {}, []
for case in CASES:
    x, X, lead, share = data[case]
    r = hd.fit(x, FAMILY, X=X, lower=LOWER, upper=UPPER); fits[case] = r
    t0 = np.r_[r["beta"], r["theta"][-1], np.zeros(len(NONREF))]
    o = optimize.minimize(nll_het, t0, args=(x, X), method="L-BFGS-B")
    o = optimize.minimize(nll_het, o.x, args=(x, X), method="Nelder-Mead", options={"maxiter": 40000, "xatol": 1e-9, "fatol": 1e-11})
    ll1 = -o.fun; lr = max(2 * (ll1 - r["loglik"]), 0.0)
    sig = np.exp(o.x[X.shape[1]:][0] + np.r_[0, o.x[X.shape[1] + 1:]])
    spread.append({"Site": case[0], "Class": case[1], "n": len(x), "logLik_shift_only": r["loglik"], "logLik_shift_and_spread": ll1,
                   "LR_stat": lr, "df": len(NONREF), "LR_p": stats.chi2.sf(lr, len(NONREF)),
                   "AIC_shift_only": r["AIC"], "AIC_shift_and_spread": 2 * (r["k"] + len(NONREF)) - 2 * ll1,
                   "sigma_shared": r["shape"][0], **{f"sigma_{c}": s_ for c, s_ in zip(LEADERS, sig)}})
spread = pd.DataFrame(spread)
spread["dAIC_spread_minus_shift"] = spread.AIC_shift_and_spread - spread.AIC_shift_only
print(spread.round(3).to_string(index=False))

### Extent of effect: each leader against the main distribution

In [ ]:
def tcdf(t, mu, s):  return hd.trunc_cdf(t, mu, FAMILY, [s], LOWER, UPPER)
def tpdf(t, mu, s):  return hd.trunc_pdf(t, mu, FAMILY, [s], LOWER, UPPER)

def summaries(beta, s, share):
    # leader-specific and mixture (main) summaries for one parameter vector
    mus = beta[0] + np.r_[0.0, beta[1:]]
    C = np.array([tcdf(GRID, m, s) for m in mus]); P = np.array([tpdf(GRID, m, s) for m in mus])
    Cm, Pm = share @ C, share @ P
    q = lambda cdfv, u: np.interp(u, cdfv, GRID)
    out = {"main": {"median": q(Cm, .5), "mean": np.trapezoid(GRID * Pm, GRID), "pshort": np.interp(SHORT, GRID, Cm),
                    "q10": q(Cm, .1), "q90": q(Cm, .9)}}
    for c, Ci, Pi in zip(LEADERS, C, P):
        out[c] = {"median": q(Ci, .5), "mean": np.trapezoid(GRID * Pi, GRID), "pshort": np.interp(SHORT, GRID, Ci),
                  "q10": q(Ci, .1), "q90": q(Ci, .9), "KS": np.abs(Ci - Cm).max()}
    return out, C, P, Cm, Pm

rng = np.random.default_rng(SEED)
rows, curves, main_rows = [], {}, []
for case in CASES:
    x, X, lead, share = data[case]; r = fits[case]
    est, C, P, Cm, Pm = summaries(r["beta"], r["shape"][0], share); curves[case] = (P, Pm)
    V = hd.covariance(r, x); draws = rng.multivariate_normal(r["theta"], V, size=N_DRAW)
    sims = [summaries(t[:X.shape[1]], np.exp(t[-1]), share)[0] for t in draws]
    r01 = hd.fit(x, FAMILY, lower=LOWER, upper=UPPER)          # unconditional fit (notebook 01) as a check on the mixture
    Fa, Fb = hd.cdf(LOWER, r01["beta"][0], FAMILY, r01["shape"]), hd.cdf(UPPER, r01["beta"][0], FAMILY, r01["shape"])
    main_rows.append({"Site": case[0], "Class": case[1], "n": len(x), "obs_median_s": np.median(x), "mixture_median_s": est["main"]["median"],
                      "uncond_fit_median_s": hd.ppf(Fa + .5 * (Fb - Fa), r01["beta"][0], FAMILY, r01["shape"]),
                      "obs_mean_s": x.mean(), "mixture_mean_s": est["main"]["mean"], f"obs_P(H<{SHORT})": (x < SHORT).mean(),
                      f"mixture_P(H<{SHORT})": est["main"]["pshort"], "mixture_Q10_s": est["main"]["q10"], "mixture_Q90_s": est["main"]["q90"]})
    for c in LEADERS:
        e, m = est[c], est["main"]
        def ci(f):
            v = np.array([f(sd[c], sd["main"]) for sd in sims]); return np.percentile(v, [2.5, 97.5])
        mr = e["median"] / m["median"]; mr_ci = ci(lambda a, b: a["median"] / b["median"])
        xs = x[lead == c]
        rows.append({"Site": case[0], "Class": case[1], "Leader": c, "n_leader": len(xs), "share": share[LEADERS.index(c)],
                     "median_s": e["median"], "main_median_s": m["median"], "median_ratio": mr, "median_ratio_lo95": mr_ci[0], "median_ratio_hi95": mr_ci[1],
                     "d_median_s": e["median"] - m["median"], "d_median_pct": 100 * (mr - 1),
                     "d_mean_s": e["mean"] - m["mean"], "d_mean_pct": 100 * (e["mean"] / m["mean"] - 1),
                     **dict(zip(["d_mean_pct_lo95", "d_mean_pct_hi95"], 100 * (ci(lambda a, b: a["mean"] / b["mean"]) - 1))),
                     f"P(H<{SHORT})": e["pshort"], f"main_P(H<{SHORT})": m["pshort"], "d_P_short_pp": 100 * (e["pshort"] - m["pshort"]),
                     **dict(zip(["d_P_short_pp_lo95", "d_P_short_pp_hi95"], 100 * ci(lambda a, b: a["pshort"] - b["pshort"]))),
                     "Q10_s": e["q10"], "Q90_s": e["q90"], "KS_distance_to_main": e["KS"],
                     "obs_median_s": np.median(xs), f"obs_P(H<{SHORT})": (xs < SHORT).mean()})
shift = pd.DataFrame(rows)
shift["Effect_type"] = np.select([shift.median_ratio_hi95 < 1, shift.median_ratio_lo95 > 1],
                                 ["compresses (shorter headways)", "stretches (longer headways)"], "no clear shift")
main = pd.DataFrame(main_rows)
print(main.round(3).to_string(index=False)); print()
print(shift[["Site", "Class", "Leader", "n_leader", "median_s", "main_median_s", "median_ratio", "median_ratio_lo95", "median_ratio_hi95",
             "d_median_s", "d_P_short_pp", "d_P_short_pp_lo95", "d_P_short_pp_hi95", "KS_distance_to_main", "Effect_type"]].round(3).to_string(index=False))

### Summary by leader class across the four cases

In [ ]:
summary = (shift.pivot_table(index="Leader", columns=["Site", "Class"], values="d_median_pct", sort=False).round(1)
           .reindex(LEADERS))
summary.columns = [f"{a} {b}: median change %" for a, b in summary.columns]
types = shift.pivot_table(index="Leader", columns=["Site", "Class"], values="Effect_type", aggfunc="first", sort=False).reindex(LEADERS)
types.columns = [f"{a} {b}: effect" for a, b in types.columns]
summary = pd.concat([summary, types], axis=1).reset_index()
print(summary.to_string(index=False))

### Figure: leader-specific headway distributions against the main distribution

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(7.2, 5.8), sharex=True)
for ax, case, tag in zip(axes.ravel(), CASES, "abcd"):
    P, Pm = curves[case]
    for c, Pi in zip(LEADERS, P):
        ax.plot(GRID, Pi, color=LEADER_COL[c], lw=1.2, label=c)
    ax.plot(GRID, Pm, color="k", lw=2.0, ls="--", label="Main (all leaders)")
    ax.set_title(f"({tag}) {case[0]}, {case[1]} (n={len(data[case][0])})", loc="left", fontsize=8.5)
    ax.grid(alpha=.25, lw=.5); ax.set_ylim(bottom=0)
axes[0, 1].legend(frameon=False, fontsize=7, loc="upper right")
for ax in axes[1]: ax.set_xlabel("Time headway (s)")
for ax in axes[:, 0]: ax.set_ylabel("Density (truncated to 0.5–5 s)")
fig.tight_layout(); save_fig(fig, "leader_shifted_densities"); plt.show()

### Figure: extent of each leader's shift against the main distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.4))
ypos = {case: i for i, case in enumerate(CASES[::-1])}; off = dict(zip(LEADERS, np.linspace(0.28, -0.28, len(LEADERS))))
for c in LEADERS:
    s_ = shift[shift.Leader == c]
    y = np.array([ypos[(a, b)] for a, b in zip(s_.Site, s_.Class)]) + off[c]
    for ax, v, lo, hi in [(axes[0], s_.d_median_pct, 100 * (s_.median_ratio_lo95 - 1), 100 * (s_.median_ratio_hi95 - 1)),
                          (axes[1], s_.d_P_short_pp, s_.d_P_short_pp_lo95, s_.d_P_short_pp_hi95)]:
        ax.errorbar(v, y, xerr=[v - lo, hi - v], fmt="o", ms=3.5, color=LEADER_COL[c], elinewidth=1.1, capsize=0, label=c)
for ax, lab in zip(axes, ["Change in median headway vs main (%)", f"Change in P(H < {SHORT:g} s) vs main (pp)"]):
    ax.axvline(0, color="0.4", lw=.7); ax.set_xlabel(lab); ax.grid(axis="x", alpha=.25, lw=.5)
    ax.set_yticks(list(ypos.values())); ax.set_yticklabels([f"{a}, {b}" for a, b in ypos])
    for k in range(len(CASES) - 1): ax.axhline(k + 0.5, color="0.85", lw=.6)
axes[1].set_yticklabels([])
axes[0].set_title("(a) Shift of the centre", loc="left", fontsize=8.5); axes[1].set_title("(b) Shift in short headways", loc="left", fontsize=8.5)
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, frameon=False, fontsize=7.5, loc="lower center", ncol=5, bbox_to_anchor=(0.5, -0.04))
fig.tight_layout(rect=(0, 0.05, 1, 1)); save_fig(fig, "leader_shift_vs_main"); plt.show()

### Export

In [ ]:
out = TAB / "02b_leader_distribution_shifts.xlsx"
with pd.ExcelWriter(out) as xw:
    summary.to_excel(xw, sheet_name="summary", index=False)
    shift.to_excel(xw, sheet_name="leader_vs_main", index=False)
    main.to_excel(xw, sheet_name="main_distribution", index=False)
    spread.to_excel(xw, sheet_name="type_shift_vs_spread", index=False)
    pd.DataFrame({"Parameter": ["LOWER", "UPPER", "family", "leader_classes", "main_distribution", "short_headway_threshold_s",
                                "N_DRAW", "SEED", "data04_sha256"],
                  "Value": [LOWER, UPPER, hd.LABELS[FAMILY], json.dumps(LEADERS), "mixture of leader-specific fits, observed leader shares",
                            SHORT, N_DRAW, SEED, hashlib.sha256(DATA.read_bytes()).hexdigest()]}).to_excel(xw, sheet_name="settings", index=False)
cambria_xlsx(out); print("wrote", out.name)

## Step 3 — Other variables as time ratios: crude and leader-adjusted Weibull AFT (4 cases)

**One variable at a time:** with n = 77 in the smallest case, the variables are not entered jointly. Each is fitted in two ways, separately for each site × target class case, with the truncated Weibull AFT chosen in step 2:

- **Crude (single-variable):** the variable alone, $\log H_i = \beta_0 + \gamma\,z_i + \sigma\varepsilon_i$. This is the plain association of the variable with headway, whatever the leader.
- **Leader-adjusted:** the variable added to the leader-only model, $\log H_i = \beta_0 + \sum_c \beta_c\,\mathbb{1}[\text{leader}_i=c] + \gamma\,z_i + \sigma\varepsilon_i$ (reference leader 4W). This is the association among events with the same leader class.

Both are truncated to $0.5\le H_i\le 5$ s. Each is reported as a time ratio $e^{\hat\gamma}$ with a 95% Wald CI and a 1-df likelihood-ratio p-value against the same model without the variable, Holm-adjusted across the three variables within each case and model.

**How to read the two:** if a crude effect shrinks once leader class is added, it was carried by the leader mix (for example, faster leaders are more often four-wheelers). If it holds, it acts within leader classes.

| Variable | Coding | Time ratio means |
|---|---|---|
| Off_Centered | 1 = follower laterally offset from the leader's path | headway when offset ÷ headway when in line |
| Occupied | 1 = follower carrying passenger(s) | headway when occupied ÷ headway when empty |
| Leader speed | km/h, centred at the case mean, per 5 km/h | headway change for a leader 5 km/h faster |

**Not entered:** the follower's own speed. Headway is spacing divided by speed, so follower speed is part of the outcome; it is handled exactly by the decomposition in step 4.

**Checks:** how much each variable moves the leader time ratios of step 2 (confounding check), and whether the time ratios hold across the other candidate families (sensitivity, both models).

**Outputs:** `outputs/tables/03_covariate_time_ratios.xlsx`, and figure `covariate_time_ratios.png`.

In [ ]:
import sys; sys.path.insert(0, str(__import__('pathlib').Path.cwd()))
from style import PAL, use_style
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
from openpyxl import load_workbook
from openpyxl.styles import Font
import hwdist as hd

HERE = Path.cwd()
DATA = HERE.parent / "outputs" / "data" / "data04.xlsx"
TAB = HERE.parent / "outputs" / "tables"; TAB.mkdir(parents=True, exist_ok=True)
FIG = HERE.parent / "outputs" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
FONT = "Cambria"
plt.rcParams.update({"font.family": FONT, "mathtext.fontset": "custom", "mathtext.rm": FONT,
                     "mathtext.it": FONT + ":italic", "mathtext.bf": FONT + ":bold", "font.size": 9})
def save_fig(fig, name):
    p = FIG / f"{name}.png"; p.unlink(missing_ok=True); fig.savefig(p, dpi=600, bbox_inches="tight"); print("saved", p.name)
def cambria_xlsx(path):
    wb = load_workbook(path)
    for ws in wb.worksheets:
        for row in ws.iter_rows():
            for c in row:
                c.font = Font(name=FONT, bold=(c.row == 1), size=11)
    wb.save(path)

### Settings

In [ ]:
LOWER, UPPER = 0.5, 5.0
FAMILY = "weibull"                 # chosen in notebook 02 (checked against its settings sheet below)
_chosen02 = pd.read_excel(HERE.parent / "outputs" / "tables" / "02_leader_conditioned_aft.xlsx", sheet_name="settings").set_index("Parameter").Value["chosen_family"]
assert _chosen02 == hd.LABELS[FAMILY], f"notebook 02 chose {_chosen02}"
SENS_FAMILIES = ["lognormal", "loglogistic", "gamma", "gengamma"]
CASES = [("Shahjahanpur", "PR"), ("Shahjahanpur", "BTW"), ("Tikatuli", "PR"), ("Tikatuli", "BTW")]
LEADERS = ["4W", "BTW", "CNG", "MT_2W", "NMT"]; REF = "4W"
NONREF = [c for c in LEADERS if c != REF]
SPEED_UNIT = 5.0                   # km/h per leader-speed step
VARS = {"Off_Centered": "Off-centred following", "Occupied": "Occupied (passengers)",
        "Leader_speed_5kmh": "Leader speed (+5 km/h)"}
VAR_COL = PAL["variable"]

### Data

In [ ]:
d = pd.read_excel(DATA, sheet_name="data04")
print("data04:", d.shape, " sha256:", hashlib.sha256(DATA.read_bytes()).hexdigest()[:16])
assert d.Time_Headway_s.between(LOWER, UPPER).all()
data, desc = {}, []
for case in CASES:
    g = d[(d.Site == case[0]) & (d.Target_Class == case[1])].copy()
    g["Leader_speed_5kmh"] = (g.Leading_Speed_kmh - g.Leading_Speed_kmh.mean()) / SPEED_UNIT
    base = np.column_stack([np.ones(len(g))] + [(g.Leader_Class == c).to_numpy(float) for c in NONREF])
    z = {"Off_Centered": g.Off_Centered.to_numpy(float), "Occupied": g.Occupied.to_numpy(float),
         "Leader_speed_5kmh": g.Leader_speed_5kmh.to_numpy(float)}
    data[case] = (g.Time_Headway_s.to_numpy(float), base, z)
    desc.append({"Site": case[0], "Class": case[1], "n": len(g), "n_Off_Centered": int(g.Off_Centered.sum()),
                 "n_Occupied": int(g.Occupied.sum()), "Leader_speed_mean_kmh": g.Leading_Speed_kmh.mean(),
                 "Leader_speed_sd_kmh": g.Leading_Speed_kmh.std()})
desc = pd.DataFrame(desc); print(desc.round(2).to_string(index=False))
MODELS = {"crude": "Crude (variable alone)", "leader_adjusted": "Leader-adjusted"}
def base_X(X0, model):
    return X0[:, :1] if model == "crude" else X0

### One-at-a-time additions: crude and leader-adjusted

In [ ]:
def fit_added(x, fam, X1, r0):
    # fit base + one variable from a warm start and from the default starts; keep the higher likelihood
    p0 = r0["X"].shape[1]
    cands = [hd.fit(x, fam, X=X1, lower=LOWER, upper=UPPER, start=np.r_[r0["beta"], 0.0, r0["theta"][p0:]]),
             hd.fit(x, fam, X=X1, lower=LOWER, upper=UPPER)]
    r1 = max(cands, key=lambda r: r["loglik"])
    assert r1["loglik"] >= r0["loglik"] - 1e-6, f"added-variable fit below base ({fam})"
    return r1

def wald(r, x, j):
    V = hd.covariance(r, x); se = float(np.sqrt(max(V[j, j], 0)))
    b = float(r["beta"][j]); return b, se

rows, leader_shift = [], []
for case in CASES:
    x, X0full, z = data[case]
    for model in MODELS:
        X0 = base_X(X0full, model)
        r0 = hd.fit(x, FAMILY, X=X0, lower=LOWER, upper=UPPER)
        for v in VARS:
            X1 = np.column_stack([X0, z[v]])
            r1 = fit_added(x, FAMILY, X1, r0)
            lr = max(2 * (r1["loglik"] - r0["loglik"]), 0.0)
            b, se = wald(r1, x, X1.shape[1] - 1)
            rows.append({"Site": case[0], "Class": case[1], "Model": model, "Variable": v, "Label": VARS[v], "n": len(x),
                         "gamma": b, "SE": se, "TR": np.exp(b), "TR_lo95": np.exp(b - 1.96 * se), "TR_hi95": np.exp(b + 1.96 * se),
                         "p_Wald": 2 * stats.norm.sf(abs(b / se)), "LR_stat": lr, "LR_p": stats.chi2.sf(lr, 1),
                         "AIC_base": r0["AIC"], "AIC_with_var": r1["AIC"], "dAIC": r1["AIC"] - r0["AIC"],
                         "sigma_base": r0["shape"][0], "sigma_with_var": r1["shape"][0]})
            if model == "leader_adjusted":
                for c, a, bb in zip(NONREF, r0["beta"][1:], r1["beta"][1:1 + len(NONREF)]):
                    leader_shift.append({"Site": case[0], "Class": case[1], "Variable": v, "Leader": c,
                                         "TR_leader_base": np.exp(a), "TR_leader_with_var": np.exp(bb),
                                         "pct_change_TR": 100 * (np.exp(bb - a) - 1)})
res = pd.DataFrame(rows)
res["LR_p_Holm"] = np.nan
for _, g in res.groupby(["Site", "Class", "Model"]):
    res.loc[g.index, "LR_p_Holm"] = multipletests(g.LR_p, method="holm")[1]
leader_shift = pd.DataFrame(leader_shift)
print(res[["Site", "Class", "Model", "Label", "TR", "TR_lo95", "TR_hi95", "LR_p", "LR_p_Holm", "dAIC"]].round(4).to_string(index=False))

### Crude vs leader-adjusted, side by side

`attenuation_pct` is how much of the crude log time ratio is removed by adjusting for leader class, $100\,(1-\hat\gamma_{\text{adj}}/\hat\gamma_{\text{crude}})$. It is shown only where the crude effect is at least 2% ($|e^{\hat\gamma}-1|\ge 0.02$); below that the ratio is unstable.

In [ ]:
side = res.pivot_table(index=["Site", "Class", "Variable"], columns="Model", values=["TR", "TR_lo95", "TR_hi95", "LR_p_Holm"], sort=False)
side.columns = [f"{m}_{k}" for k, m in side.columns]
side = side.reset_index()
gc, ga = np.log(side.crude_TR), np.log(side.leader_adjusted_TR)
side["attenuation_pct"] = np.where(np.abs(side.crude_TR - 1) >= 0.02, 100 * (1 - ga / gc), np.nan)
side = side[["Site", "Class", "Variable", "crude_TR", "crude_TR_lo95", "crude_TR_hi95", "crude_LR_p_Holm",
             "leader_adjusted_TR", "leader_adjusted_TR_lo95", "leader_adjusted_TR_hi95", "leader_adjusted_LR_p_Holm", "attenuation_pct"]]
print(side.round(3).to_string(index=False))

### Check: how far each variable moves the leader time ratios

In [ ]:
shift = (leader_shift.assign(abs_pct=lambda t: t.pct_change_TR.abs())
         .groupby(["Site", "Class", "Variable"], sort=False).abs_pct.max().rename("max_abs_pct_change_leader_TR").reset_index())
print(shift.round(2).to_string(index=False))

### Sensitivity: the same time ratios under the other candidate families

In [ ]:
sens = []
for case in CASES:
    x, X0full, z = data[case]
    for model in MODELS:
        X0 = base_X(X0full, model)
        for fam in [FAMILY] + SENS_FAMILIES:
            r0 = hd.fit(x, fam, X=X0, lower=LOWER, upper=UPPER)
            for v in VARS:
                X1 = np.column_stack([X0, z[v]])
                r1 = fit_added(x, fam, X1, r0)
                b, se = wald(r1, x, X1.shape[1] - 1); lr = max(2 * (r1["loglik"] - r0["loglik"]), 0.0)
                sens.append({"Site": case[0], "Class": case[1], "Model": model, "Variable": v, "Family": hd.LABELS[fam], "TR": np.exp(b),
                             "TR_lo95": np.exp(b - 1.96 * se), "TR_hi95": np.exp(b + 1.96 * se), "LR_p": stats.chi2.sf(lr, 1),
                             "GG_Q": r1["shape"][1] if fam == "gengamma" else np.nan})
sens = pd.DataFrame(sens)
sens["weakly_identified"] = sens.GG_Q.abs() > 5      # generalized gamma on the high-|Q| ridge: time ratio not interpretable
print("weakly identified fits:", sens[sens.weakly_identified][["Site", "Class", "Model", "Variable", "GG_Q"]].to_dict("records"))
print(sens.pivot_table(index=["Site", "Class", "Model", "Variable"], columns="Family", values="TR", sort=False).round(3).to_string())

### Figure: time ratios of the added variables, crude and leader-adjusted (Weibull)

In [ ]:
from matplotlib.lines import Line2D
fig, axes = plt.subplots(2, 2, figsize=(6.2, 4.8), sharey=True, sharex=True)
vlist = list(VARS); ypos = np.arange(len(vlist))[::-1]; OFF = {"crude": 0.14, "leader_adjusted": -0.14}
DASH = (0, (2, 1.2))
YLAB = {"Off_Centered": "Off-centred\nfollowing", "Occupied": "Occupied\n(passengers)", "Leader_speed_5kmh": "Leader speed\n(+5 km/h)"}
for ax, case, tag in zip(axes.ravel(), CASES, "abcd"):
    ax.axvline(1, color="0.4", lw=.7)
    for model in MODELS:
        g = res[(res.Site == case[0]) & (res.Class == case[1]) & (res.Model == model)].set_index("Variable").loc[vlist]
        for y, v in zip(ypos + OFF[model], vlist):
            adj = model == "leader_adjusted"
            ax.plot([g.loc[v, "TR_lo95"], g.loc[v, "TR_hi95"]], [y, y], color=VAR_COL[v], lw=1.3, ls="-" if adj else DASH)
            ax.plot(g.loc[v, "TR"], y, "o", ms=4, color=VAR_COL[v], mfc=VAR_COL[v] if adj else "white", mew=1.1)
    ax.set_title(f"({tag}) {case[0]}, {case[1]} (n={len(data[case][0])})", loc="left", fontsize=8.5)
    ax.set_xscale("log"); ax.set_xlim(0.33, 1.7); ax.set_xticks([0.5, 0.7, 1, 1.4]); ax.set_xticklabels(["0.5", "0.7", "1", "1.4"])
    ax.minorticks_off(); ax.grid(axis="x", alpha=.25, lw=.5)
for ax in axes[:, 0]:
    ax.set_yticks(ypos); ax.set_yticklabels([YLAB[v] for v in vlist], fontsize=8.5)
hand = [Line2D([], [], color="0.25", ls=DASH, marker="o", mfc="white", ms=4, label="Crude (variable alone)"),
        Line2D([], [], color="0.25", ls="-", marker="o", ms=4, label="Leader-adjusted")]
fig.tight_layout(rect=(0, 0.1, 1, 1))
fig.text(0.55, 0.065, "Time ratio, truncated Weibull AFT (95% CI, log scale)", ha="center", fontsize=8.5)
fig.legend(handles=hand, frameon=False, fontsize=7.5, ncol=2, loc="lower center", bbox_to_anchor=(0.55, -0.01))
save_fig(fig, "covariate_time_ratios"); plt.show()

### Export

In [ ]:
out = TAB / "03_covariate_time_ratios.xlsx"
with pd.ExcelWriter(out) as xw:
    side.to_excel(xw, sheet_name="crude_vs_adjusted", index=False)
    res.to_excel(xw, sheet_name="time_ratios", index=False)
    desc.to_excel(xw, sheet_name="variables_by_case", index=False)
    leader_shift.to_excel(xw, sheet_name="leader_TR_shift", index=False)
    sens.to_excel(xw, sheet_name="family_sensitivity", index=False)
    pd.DataFrame({"Parameter": ["LOWER", "UPPER", "family", "reference_leader", "leader_speed_unit_kmh", "leader_speed_centring",
                                "sensitivity_families", "data04_sha256"],
                  "Value": [LOWER, UPPER, hd.LABELS[FAMILY], REF, SPEED_UNIT, "case mean", json.dumps(SENS_FAMILIES),
                            hashlib.sha256(DATA.read_bytes()).hexdigest()]}).to_excel(xw, sheet_name="settings", index=False)
cambria_xlsx(out); print("wrote", out.name)